In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import shap
import matplotlib.pyplot as plt

In [3]:
file_path = "data/SLIIT_IT_Student_Data_Template_filled-updated.csv"   # <- change this to your real file
df = pd.read_csv(file_path)

print("First 5 rows:")
print(df.head())
print("\nColumns:")
print(df.columns)


First 5 rows:
   Gender District_Location Education_Stream  \
0    Male           Colombo       Bio Stream   
1  Female           Gampaha     Maths Stream   
2    Male             Kandy       Bio Stream   
3  Female             Galle     Maths Stream   
4    Male            Matara  Commerce Stream   

                                         Soft_Skills  \
0           Communication, Teamwork, Time Management   
1     Leadership, Problem Solving, Critical Thinking   
2            Public Speaking, Teamwork, Adaptability   
3  Communication, Presentation Skills, Emotional ...   
4   Time Management, Work Ethic, Attention to Detail   

                                           Key_Skils Current_semester  GPA  \
0  Java, Python, Spring Boot, MySQL, Git, RESTful...             1Y1S  1.0   
1  SQL, Python, Pandas, Power BI, Tableau, Excel ...             1Y2S  1.3   
2  HTML5, CSS3, JavaScript ES6+, React.js, Tailwi...             2Y1S  2.1   
3  Java, Spring Boot, PostgreSQL, REST APIs, Mav

In [4]:
# 3.1 Remove unwanted 'Unnamed' columns from Excel
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# 3.2 Define features and target
target_col = "Experties_Preferred_Career_Field"
feature_cols = ["Soft_Skills", "Key_Skils", "Current_semester", "GPA"]

# 3.3 Drop rows with missing target
df = df.dropna(subset=[target_col])

# 3.4 Handle missing + types in feature columns
df["Soft_Skills"] = df["Soft_Skills"].fillna("")
df["Key_Skils"] = df["Key_Skils"].fillna("")

df["GPA"] = pd.to_numeric(df["GPA"], errors="coerce")
df = df.dropna(subset=["GPA"])

df["Current_semester"] = df["Current_semester"].astype(str)

print("\nRows after cleaning:", len(df))
print("\nDtypes:")
print(df[feature_cols + [target_col]].dtypes)



Rows after cleaning: 499

Dtypes:
Soft_Skills                          object
Key_Skils                            object
Current_semester                     object
GPA                                 float64
Experties_Preferred_Career_Field     object
dtype: object


In [5]:
X = df[feature_cols].copy()
y = df[target_col].copy()

# Encode string career labels → integer classes
label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)

print("\nOriginal labels:", sorted(y.unique()))
print("Encoded label values:", np.unique(y_encoded))



Original labels: ['AI / Machine Learning Engineer', 'Cybersecurity / Network Engineer', 'Data Science & Analytics', 'DevOps / Cloud Engineer', 'Frontend / Web Developer', 'IT / Technology Generalist', 'Mobile App Developer', 'Software Engineer / Backend', 'UI/UX Designer']
Encoded label values: [0 1 2 3 4 5 6 7 8]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain size:", X_train.shape[0])
print("Test size:", X_test.shape[0])



Train size: 399
Test size: 100


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("soft_tfidf", TfidfVectorizer(), "Soft_Skills"),
        ("key_tfidf", TfidfVectorizer(), "Key_Skils"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Current_semester"]),
        ("num", StandardScaler(), ["GPA"])
    ]
)


In [8]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),  # 2 hidden layers
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)


In [9]:
mlp_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("mlp", mlp_model)
])


In [10]:
print("Training MLP model...")
mlp_clf.fit(X_train, y_train)
print("✅ MLP training completed.")


Training MLP model...
✅ MLP training completed.


In [11]:
y_pred_mlp = mlp_clf.predict(X_test)

mlp_acc = accuracy_score(y_test, y_pred_mlp)
print(f"\nMLP Accuracy: {mlp_acc:.4f}")

print("\nMLP Classification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=label_enc.classes_))

print("\nMLP Confusion Matrix (encoded labels):")
print(confusion_matrix(y_test, y_pred_mlp))



MLP Accuracy: 0.8700

MLP Classification Report:
                                  precision    recall  f1-score   support

  AI / Machine Learning Engineer       1.00      0.86      0.92         7
Cybersecurity / Network Engineer       1.00      1.00      1.00         3
        Data Science & Analytics       0.76      0.93      0.83        27
         DevOps / Cloud Engineer       0.91      0.87      0.89        23
        Frontend / Web Developer       0.87      1.00      0.93        13
      IT / Technology Generalist       1.00      0.17      0.29         6
            Mobile App Developer       1.00      1.00      1.00         7
     Software Engineer / Backend       0.80      0.67      0.73         6
                  UI/UX Designer       1.00      1.00      1.00         8

                        accuracy                           0.87       100
                       macro avg       0.93      0.83      0.84       100
                    weighted avg       0.88      0.87      0

In [12]:
# Sample a smaller background set for SHAP (speed)
X_train_bg = X_train.sample(n=min(100, len(X_train)), random_state=42)

# Transform using the same preprocessor from pipeline
X_bg_trans = mlp_clf.named_steps["preprocess"].transform(X_train_bg)
X_test_trans = mlp_clf.named_steps["preprocess"].transform(X_test)

# Convert to dense arrays (KernelExplainer prefers dense)
if not isinstance(X_bg_trans, np.ndarray):
    X_bg_dense = X_bg_trans.toarray()
else:
    X_bg_dense = X_bg_trans

if not isinstance(X_test_trans, np.ndarray):
    X_test_dense = X_test_trans.toarray()
else:
    X_test_dense = X_test_trans

# Get feature names
feature_names = mlp_clf.named_steps["preprocess"].get_feature_names_out()

print("\nBackground shape:", X_bg_dense.shape)
print("Test transformed shape:", X_test_dense.shape)
print("Num features:", len(feature_names))



Background shape: (100, 729)
Test transformed shape: (100, 729)
Num features: 729


In [13]:
# Function that takes transformed X and returns predicted probabilities
def mlp_predict_proba_transformed(X_trans):
    """
    X_trans: already preprocessed, transformed feature matrix.
    """
    return mlp_clf.named_steps["mlp"].predict_proba(X_trans)

# Build SHAP explainer using background data
explainer_mlp = shap.KernelExplainer(
    mlp_predict_proba_transformed,
    X_bg_dense
)


In [14]:
# Limit to first N test samples for SHAP (to keep it fast)
N_SAMPLES = min(50, X_test_dense.shape[0])
X_test_shap = X_test_dense[:N_SAMPLES]
y_test_shap = y_test[:N_SAMPLES]

# Calculate SHAP values
# For multiclass, this returns a list [array_for_class0, array_for_class1, ...]
print("Computing SHAP values for MLP (this can take a bit)...")
shap_values_mlp = explainer_mlp.shap_values(X_test_shap)
print("✅ SHAP values computed.")


Computing SHAP values for MLP (this can take a bit)...


100%|██████████| 50/50 [01:09<00:00,  1.39s/it]

✅ SHAP values computed.


In [15]:
for i, sv in enumerate(shap_values_mlp):
    print(f"Class {i} SHAP shape:", sv.shape)


Class 0 SHAP shape: (729, 9)
Class 1 SHAP shape: (729, 9)
Class 2 SHAP shape: (729, 9)
Class 3 SHAP shape: (729, 9)
Class 4 SHAP shape: (729, 9)
Class 5 SHAP shape: (729, 9)
Class 6 SHAP shape: (729, 9)
Class 7 SHAP shape: (729, 9)
Class 8 SHAP shape: (729, 9)
Class 9 SHAP shape: (729, 9)
Class 10 SHAP shape: (729, 9)
Class 11 SHAP shape: (729, 9)
Class 12 SHAP shape: (729, 9)
Class 13 SHAP shape: (729, 9)
Class 14 SHAP shape: (729, 9)
Class 15 SHAP shape: (729, 9)
Class 16 SHAP shape: (729, 9)
Class 17 SHAP shape: (729, 9)
Class 18 SHAP shape: (729, 9)
Class 19 SHAP shape: (729, 9)
Class 20 SHAP shape: (729, 9)
Class 21 SHAP shape: (729, 9)
Class 22 SHAP shape: (729, 9)
Class 23 SHAP shape: (729, 9)
Class 24 SHAP shape: (729, 9)
Class 25 SHAP shape: (729, 9)
Class 26 SHAP shape: (729, 9)
Class 27 SHAP shape: (729, 9)
Class 28 SHAP shape: (729, 9)
Class 29 SHAP shape: (729, 9)
Class 30 SHAP shape: (729, 9)
Class 31 SHAP shape: (729, 9)
Class 32 SHAP shape: (729, 9)
Class 33 SHAP shape:

In [17]:
print("Class index mapping:")
for idx, label in enumerate(label_enc.classes_):
    print(idx, "->", label)


Class index mapping:
0 -> AI / Machine Learning Engineer
1 -> Cybersecurity / Network Engineer
2 -> Data Science & Analytics
3 -> DevOps / Cloud Engineer
4 -> Frontend / Web Developer
5 -> IT / Technology Generalist
6 -> Mobile App Developer
7 -> Software Engineer / Backend
8 -> UI/UX Designer


In [19]:
i = 0  # sample index within the SHAP subset

x_row = X_test_shap[i:i+1]  # shape (1, n_features)

# SHAP values for this row (per class)
shap_row = [sv[i:i+1, :] for sv in shap_values_mlp]

# Predicted encoded class
pred_encoded = mlp_clf.predict(
    X_test.iloc[:N_SAMPLES].iloc[i:i+1]
)[0]
pred_label = label_enc.inverse_transform([pred_encoded])[0]

print("Predicted career for this student:", pred_label)

# Show force plot for that predicted class
class_idx = pred_encoded  # because y_encoded used LabelEncoder indices

shap.force_plot(
    explainer_mlp.expected_value[class_idx],
    shap_row[class_idx],
    x_row,
    feature_names=feature_names
)


Predicted career for this student: Mobile App Developer


DimensionError: Length of features is not equal to the length of shap_values!